# Quickstart: Querying PDF With Astra and LangChain

### A question-answering demo using Astra DB and LangChain, powered by Vector Search

#### Pre-requisites:

You need a **_Serverless Cassandra with Vector Search_** database on [Astra DB](https://astra.datastax.com) to run this demo. As outlined in more detail [here](https://docs.datastax.com/en/astra-serverless/docs/vector-search/quickstart.html#_prepare_for_using_your_vector_database), you should get a DB Token with role _Database Administrator_ and copy your Database ID: these connection parameters are needed momentarily.

You also need an [OpenAI API Key](https://cassio.org/start_here/#llm-access) for this demo to work.

#### What you will do:

- Setup: import dependencies, provide secrets, create the LangChain vector store;
- Run a Question-Answering loop retrieving the relevant headlines and having an LLM construct the answer.

Install the required dependencies:

In [ ]:
!pip install -q cassio datasets langchain openai tiktoken

Import the packages you'll need:

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Load credentials from .env file (never hardcode secrets in notebooks)
# Copy .env.example to .env and fill in your actual values
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_ID = os.getenv("ASTRA_DB_ID")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
import cassio
from langchain_openai import OpenAI, OpenAIEmbeddings  
from langchain_community.vectorstores import Cassandra
from langchain_text_splitters import CharacterTextSplitter 
from datasets import load_dataset
from PyPDF2 import PdfReader
from typing_extensions import Concatenate

# PDF reader
pdfreader = PdfReader('budget_speech.pdf')
raw_text = ''
for page in pdfreader.pages:
    content = page.extract_text()
    if content:
        raw_text += content


cassio.init(token=ASTRA_DB_APPLICATION_TOKEN, database_id=ASTRA_DB_ID)
llm = OpenAI(openai_api_key=OPENAI_API_KEY)
embedding = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

astra_vector_store = Cassandra(
    embedding=embedding,
    table_name="qa_mini_demo",
    session=None,
    keyspace=None,
)


text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=800,
    chunk_overlap=200,
    length_function=len,
)
texts = text_splitter.split_text(raw_text)

astra_vector_store.add_texts(texts[:50])
print("Inserted %i headlines." % len(texts[:50]))
retriever = astra_vector_store.as_retriever(search_kwargs={"k": 5})

docs = retriever.invoke("budget deficit")
print(f"Retrieved {len(docs)} docs")l

Inserted 50 headlines.
Retrieved 5 docs


In [18]:
from langchain_classic.chains import RetrievalQA

# RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm, 
    retriever=astra_vector_store.as_retriever()
)
result = qa_chain.invoke("What is the budget speech about?")
print(result["result"])

 The budget speech is about the Government of India's budget for the fiscal year 2025-2026, presented by Minister of Finance Nirmala Sitharaman. It outlines the government's efforts to accelerate growth, secure inclusive development, invigorate private sector investments, uplift household sentiments, and enhance spending power of India's rising middle class. The proposed development measures focus on ten broad areas, including agriculture, rural prosperity, inclusive growth, manufacturing, MSMEs, employment, investing in people, economy and innovation, securing energy supplies, promoting exports, and nurturing innovation.


In [20]:
first_question = True
while True:
    if first_question:
        query_text = input("\nEnter your question (or type 'quit' to exit): ").strip()
    else:
        query_text = input("\nWhat's your next question (or type 'quit' to exit): ").strip()

    if query_text.lower() == "quit":
        break

    if query_text == "":
        continue

    first_question = False

    print("\nQUESTION: \"%s\"" % query_text)
    answer = qa_chain.invoke(query_text)
    print("ANSWER: \"%s\"\n" % answer)

    print("FIRST DOCUMENTS BY RELEVANCE:")
    for doc, score in astra_vector_store.similarity_search_with_score(query_text, k=4):
        print("    [%0.4f] \"%s ...\"" % (score, doc.page_content[:84]))


QUESTION: "What is the budget speech about"
ANSWER: "{'query': 'What is the budget speech about', 'result': " The budget speech is about the Government of India's budget for the financial year 2025-2026, presented by the Minister of Finance, Nirmala Sitharaman. It outlines the government's efforts to accelerate growth, secure inclusive development, invigorate private sector investments, uplift household sentiments, and enhance spending power of India's rising middle class. The speech also mentions the government's aspiration for a developed India and the proposed development measures in various areas such as agriculture, rural prosperity, inclusive growth, manufacturing, employment, investment, exports, and innovation. "}"

FIRST DOCUMENTS BY RELEVANCE:
    [0.9197] "GOVERNMENT OF INDIA
BUDGET 2025-2026
SPEECH
OF
NIRMALA SITHARAMAN
MINISTER OF FINANC ..."
    [0.9176] "Minister of Finance  
February 1 , 202 5 
Hon’ble Speaker,  
 I present the Budget f ..."
    [0.8989] "realize ‘Sabk